# 🎬 ICAN CLIP

**Auto Short-Form Generator** — mirip Opus Clip

## Pipeline
```
YouTube URL
    ↓ yt-dlp
Faster Whisper Large-v3
    ↓ Pyannote Diarization
Gemini 2.5 Flash
    ↓ Hook Detection
Virality Scoring
    ↓ YOLO Face Tracking
Word Subtitle
    ↓ Auto Reframe 9:16
Export Shorts
```

## 1. Install Dependencies

In [ ]:
!pip install -q yt-dlp faster-whisper google-generativeai moviepy opencv-python-headless \
    ultralytics pyannote.audio python-dotenv Pillow tqdm

# FFmpeg (Colab biasanya sudah ada)
import shutil
print('ffmpeg:', shutil.which('ffmpeg'))

## 2. API Keys

In [ ]:
import os
from google.colab import userdata

os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
os.environ['HUGGINGFACE_TOKEN'] = userdata.get('HUGGINGFACE_TOKEN')  # untuk Pyannote

print('✅ API keys loaded')

## 3. Clone / Mount Project

In [ ]:
# Opsi A: Clone dari GitHub
# !git clone https://github.com/YOUR_REPO/AI-Clipping-Software.git
# %cd AI-Clipping-Software

# Opsi B: Upload manual — pastikan semua file services/ ada
import os
print('CWD:', os.getcwd())
print('Files:', os.listdir('.'))

## 4. Konfigurasi

In [ ]:
YOUTUBE_URL = "https://www.youtube.com/watch?v=VIDEO_ID"  # ← ganti URL
NUM_CLIPS = 5
MIN_DURATION = 15   # detik
MAX_DURATION = 60     # detik
CAPTION_STYLE = 'bright_yellow'  # clean_white, bright_yellow, neon_cyan, dll
USE_SUBTITLES = True
LANGUAGE = None  # None = auto-detect, atau 'id', 'en'

## 5. Jalankan ICAN CLIP Pipeline

In [ ]:
from services.ican_processor import IcanProcessor
from config_ican import OUTPUT_DIR

processor = IcanProcessor(
    caption_style=CAPTION_STYLE,
    use_subtitles=USE_SUBTITLES,
)

outputs, title, metadata = processor.process(
    url=YOUTUBE_URL,
    num_clips=NUM_CLIPS,
    min_duration=MIN_DURATION,
    max_duration=MAX_DURATION,
    language=LANGUAGE,
)

print(f"\n🎉 Selesai! {len(outputs)} Shorts dari: {title}")
for i, meta in enumerate(metadata, 1):
    print(f"  {i}. ⭐{meta['virality_score']}/100 | 🪝{meta['hook_type']} | {meta['title']}")

## 6. Preview & Download

In [ ]:
from IPython.display import Video, display
from pathlib import Path

for path in outputs:
    p = Path(path)
    if p.exists():
        print(f"📹 {p.name} ({p.stat().st_size / 1024 / 1024:.1f} MB)")
        display(Video(str(p), width=360, height=640))

# Download semua klip (Colab)
from google.colab import files
for path in outputs:
    if Path(path).exists():
        files.download(path)